# 🛠️ Argo on Colab — Qwen3-14B-abliterated (رایگان، GPU)

این نوت‌بوک یه سرور **Argo** کامل بالا میاره:

- **Qwen3-14B-abliterated** با llama.cpp روی GPU (T4)
- OpenAI-compatible API روی پورت 8080
- **Argo UI** (چت + agent) روی پورت 8000
- تونل عمومی Cloudflare (بدون ثبت‌نام)

## راه‌اندازی

1. **Runtime → Change runtime type → T4 GPU**
2. سلول‌ها رو به ترتیب **Shift+Enter** اجرا کن
3. آخرین سلول یه URL عمومی میده — روش بزن و چت کن

## چقدر طول می‌کشه؟

- بار اول: ~5-7 دقیقه (نصب + دانلود 9GB مدل)
- بار دوم (با Drive cache): ~1-2 دقیقه

## مصرف منابع

- VRAM: ~12GB (از 16GB T4)
- RAM: ~4GB
- Disk: ~10GB

## 1) GPU check + setup

In [ ]:
!nvidia-smi | head -10
!echo '---'
!free -h | head -3
!df -h / | tail -1

## 2) نصب llama.cpp (از سورس با CUDA)

نصب از سورس بهترین سازگاری رو با GPU میده. حدود 2-3 دقیقه طول می‌کشه.

In [ ]:
import os, subprocess, sys, time

os.chdir('/content')

# Only build if not already present
if not os.path.exists('/content/llama.cpp/build/bin/llama-server'):
    print('⏬ Cloning llama.cpp...')
    subprocess.run(['git', 'clone', '--depth=1', 'https://github.com/ggerganov/llama.cpp'], check=True)
    
    os.chdir('/content/llama.cpp')
    
    print('🔨 Building with CUDA (T4 sm_75)...')
    t0 = time.time()
    subprocess.run(['cmake', '-B', 'build', '-DGGML_CUDA=ON', '-DCMAKE_CUDA_ARCHITECTURES=75'], check=True)
    subprocess.run(['cmake', '--build', 'build', '--config', 'Release', '-j'], check=True)
    print(f'✅ Build done in {time.time()-t0:.0f}s')
else:
    print('✅ llama.cpp already built')
    os.chdir('/content/llama.cpp')
    
subprocess.run(['ls', '-la', 'build/bin/'])
subprocess.run(['./build/bin/llama-server', '--version'])

## 3) دانلود Qwen3-14B-abliterated (Q4_K_M)

مدل رو روی Google Drive کش می‌کنیم تا دفعه بعد سریع باشه.

**اندازه:** ~9GB
**منبع:** `huihui-ai/Qwen3-14B-abliterated-GGUF` (bartowski conversion — باکیفیت)

In [ ]:
import os, subprocess, time
from pathlib import Path

# Cache on Drive for fast re-runs (optional — falls back to /content if no Drive)
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    CACHE = Path('/content/drive/MyDrive/argo_models')
    print('💾 Drive cache:', CACHE)
except Exception:
    CACHE = Path('/content/models')
    print('💾 Local cache:', CACHE)

CACHE.mkdir(parents=True, exist_ok=True)

MODEL_FILE = 'qwen3-14b-abliterated.Q4_K_M.gguf'
LOCAL = CACHE / MODEL_FILE
EXPECTED_SIZE = 9_000_000_000  # ~9GB

if LOCAL.exists() and LOCAL.stat().st_size > EXPECTED_SIZE * 0.95:
    print(f'✅ Model already cached: {LOCAL} ({LOCAL.stat().st_size/1e9:.2f} GB)')
else:
    print(f'⏬ Downloading {MODEL_FILE} (~9GB, first run only)...')
    t0 = time.time()
    # Use huggingface_hub for fast parallel download
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub'
    ], check=True)
    
    from huggingface_hub import hf_hub_download
    hf_hub_download(
        repo_id='huihui-ai/Qwen3-14B-abliterated-GGUF',
        filename='qwen3-14b-abliterated.Q4_K_M.gguf',
        local_dir=str(CACHE),
    )
    print(f'✅ Downloaded in {time.time()-t0:.0f}s')
    
# Verify
print(f'📁 Model: {LOCAL}')
print(f'📊 Size:  {LOCAL.stat().st_size/1e9:.2f} GB')

## 4) دانلود + نصب Argo (از GitHub)

اگه Argo رو تغییر دادی و خواستی از repo خودت بگیری، `ARGO_REPO` رو عوض کن.

In [ ]:
import os, subprocess

os.chdir('/content')
ARGO_REPO = os.environ.get('ARGO_REPO', 'https://github.com/minam67889-bit/argo.git')
ARGO_BRANCH = os.environ.get('ARGO_BRANCH', 'main')

if not os.path.exists('/content/argo/app/main.py'):
    print(f'⏬ Cloning Argo from {ARGO_REPO}...')
    subprocess.run(['git', 'clone', '--depth=1', '-b', ARGO_BRANCH, ARGO_REPO, 'argo'], check=True)
else:
    print('🔄 Updating Argo...')
    os.chdir('/content/argo')
    subprocess.run(['git', 'pull'], check=False)
    os.chdir('/content')

os.chdir('/content/argo')
print('📦 Installing Python deps...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
print('✅ Argo ready')

## 5) راه‌اندازی llama-server (مدل)

مدل رو روی GPU می‌گذاره با `n_gpu_layers=-1` (همه لایه‌ها رو GPU). context = 12K تا با T4 جا بشه و agent بتونه history بلند داشته باشه.

In [ ]:
import os, subprocess, time, requests, signal
from pathlib import Path

LLAMA_PORT = 8080
CONTEXT_SIZE = 12288  # 12K context (به جای پیش‌فرض 4K)
MODEL_PATH = str(LOCAL)  # from previous cell
LLAMA_BIN = '/content/llama.cpp/build/bin/llama-server'

assert Path(LLAMA_BIN).exists(), f'llama-server not found at {LLAMA_BIN}'
assert Path(MODEL_PATH).exists(), f'Model not found at {MODEL_PATH}'

# Kill any previous instance
subprocess.run(['pkill', '-9', '-f', 'llama-server'], check=False)
time.sleep(2)

print(f'🚀 Starting llama-server on port {LLAMA_PORT}...')
print(f'   Model: {MODEL_PATH}')
print(f'   Context: {CONTEXT_SIZE} tokens')
print(f'   GPU layers: all (offload to T4)')
print()

log = open('/content/llama.log', 'w')
proc = subprocess.Popen([
    LLAMA_BIN,
    '-m', MODEL_PATH,
    '--host', '127.0.0.1',
    '--port', str(LLAMA_PORT),
    '-c', str(CONTEXT_SIZE),
    '-ngl', '999',           # همه لایه‌ها رو GPU
    '-np', '2',              # 2 slot موازی (برای agent)
    '--chat-template', 'chatml',
    '--jinja',               # برای tool calling (Qwen3-style)
    '--reasoning-format', 'auto',  # Qwen3 thinking
    '--no-webui',            # غیرفعال: Argo UI خودشو داره
    '--flash-attn',          # سرعت بیشتر رو T4
    '--cont-batching',
    '--log-disable',
], stdout=log, stderr=subprocess.STDOUT, start_new_session=True)

print(f'   PID: {proc.pid}')
print('⏳ Waiting for server to load (30-60s)...')

# Wait for /health
for i in range(120):
    try:
        r = requests.get(f'http://127.0.0.1:{LLAMA_PORT}/health', timeout=2)
        if r.status_code == 200:
            print(f'\n✅ llama-server ready after {i}s!')
            break
    except Exception:
        pass
    time.sleep(1)
else:
    print('\n❌ Server failed to start. Last 30 log lines:')
    subprocess.run(['tail', '-30', '/content/llama.log'])
    raise SystemExit(1)

# Test it
print('\n🧪 Quick test...')
r = requests.post(
    f'http://127.0.0.1:{LLAMA_PORT}/v1/chat/completions',
    json={
        'model': 'qwen3-14b-abliterated',
        'messages': [{'role': 'user', 'content': 'سلام، یک جمله بگو.'}],
        'max_tokens': 100,
    },
    timeout=120,
)
r.raise_for_status()
data = r.json()
print(f'✅ Response: {data["choices"][0]["message"]["content"]!r}')
print(f'   Tokens: {data["usage"]}')

## 6) راه‌اندازی Argo UI

Argo به llama-server وصل میشه و UI وب میده روی پورت 8000.

In [ ]:
import os, subprocess, time, requests

ARGO_PORT = 8000
LLAMA_PORT = 8080  # from prev cell

# Kill any previous instance
subprocess.run(['pkill', '-9', '-f', 'app.main'], check=False)
time.sleep(2)

# Set env so Argo points to the local llama-server
env = os.environ.copy()
env['LLM_API_KEY'] = 'sk-no-key-required'
env['LLM_BASE_URL'] = f'http://127.0.0.1:{LLAMA_PORT}/v1'
env['LLM_MODEL'] = 'qwen3-14b-abliterated'
env['AGENT_TEMPERATURE'] = '0.2'
env['AGENT_MAX_TOKENS'] = '4096'
env['AGENT_MAX_STEPS'] = '30'
env['ARGO_PORT'] = str(ARGO_PORT)
env['ARGO_HOST'] = '127.0.0.1'
env['ARGO_WORKSPACE'] = '/content/workspace'
os.makedirs('/content/workspace', exist_ok=True)

os.chdir('/content/argo')
print('🚀 Starting Argo on port', ARGO_PORT, '...')
argo_log = open('/content/argo.log', 'w')
argo_proc = subprocess.Popen(
    [sys.executable, '-m', 'app.main'],
    env=env,
    stdout=argo_log,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)
print(f'   PID: {argo_proc.pid}')
print('⏳ Waiting for Argo to come up...')

for i in range(30):
    try:
        r = requests.get(f'http://127.0.0.1:{ARGO_PORT}/api/health', timeout=2)
        if r.status_code == 200:
            print(f'\n✅ Argo ready after {i}s')
            h = r.json()
            print(f'   Model:  {h["model"]}')
            print(f'   API:    {h["base_url"]}')
            print(f'   Key:    {"set" if h["has_api_key"] else "not set"}')
            break
    except Exception:
        pass
    time.sleep(1)
else:
    print('❌ Argo failed. Log:')
    subprocess.run(['tail', '-30', '/content/argo.log'])
    raise SystemExit(1)

## 7) Cloudflare Tunnel — URL عمومی میده

بدون ثبت‌نام، سریع، ناشناس. فقط پورت Argo (8000) رو expose می‌کنیم چون UI همه چیزو داره.

In [ ]:
import os, subprocess, time, re, requests

TUNNEL_PORT = 8000  # Argo UI port

# Download cloudflared
if not os.path.exists('/usr/local/bin/cloudflared'):
    print('⏬ Downloading cloudflared...')
    subprocess.run([
        'wget', '-q', '-O', '/tmp/cloudflared',
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'
    ], check=True)
    subprocess.run(['chmod', '+x', '/tmp/cloudflared'], check=True)
    os.makedirs('/usr/local/bin', exist_ok=True)
    subprocess.run(['mv', '/tmp/cloudflared', '/usr/local/bin/cloudflared'], check=True)
    print('✅ cloudflared installed')

# Kill any previous tunnel
subprocess.run(['pkill', '-9', '-f', 'cloudflared'], check=False)
time.sleep(1)

print('🌐 Starting Cloudflare tunnel on Argo port', TUNNEL_PORT, '...')
tunnel_log = open('/content/tunnel.log', 'w')
tunnel_proc = subprocess.Popen([
    'cloudflared', 'tunnel', '--no-autoupdate',
    '--url', f'http://127.0.0.1:{TUNNEL_PORT}',
    '--metrics', '127.0.0.1:0',
], stdout=tunnel_log, stderr=subprocess.STDOUT, start_new_session=True)

print('⏳ Waiting for tunnel URL...')
url = None
for i in range(60):
    time.sleep(1)
    try:
        text = tunnel_log.read() if not tunnel_log.closed else open('/content/tunnel.log').read()
    except Exception:
        text = open('/content/tunnel.log').read()
    m = re.search(r'(https://[a-z0-9-]+\.trycloudflare\.com)', text)
    if m:
        url = m.group(1)
        break

if not url:
    print('❌ Tunnel failed. Log:')
    print(open('/content/tunnel.log').read())
    raise SystemExit(1)

print()
print('=' * 70)
print('✅ Argo is LIVE!')
print('=' * 70)
print()
print(f'🌐 URL: {url}')
print()
print('Settings for OpenAI-compatible clients (Cline, Aider, OpenHands, …):')
print(f'   Base URL: {url}/v1')
print(f'   API Key:  sk-no-key-required')
print(f'   Model:    qwen3-14b-abliterated')
print()
print('The browser UI is at the URL above. Open it in any browser.')
print()
print('=' * 70)
print('To stop everything: Runtime → Interrupt execution, then:')
print('  pkill -9 -f llama-server; pkill -9 -f app.main; pkill -9 -f cloudflared')
print('=' * 70)

## 🎉 تموم! حالا میتونی استفاده کنی

روی URL بالا بزن و:

- **حالت چت**: گفتگوی آزاد با Qwen3-14B-abliterated (uncensored)
- **حالت ایجنت**: فایل آپلود کن، تسک بده (مثل «این پروژه رو بررسی کن»)، ایجنت با bash + فایل کار می‌کنه

### اتصال سایر ابزارها به همین سرور

هر ابزاری که OpenAI-compatible باشه می‌تونه وصل بشه:

**Cline / Continue / Roo Code:**
```
Base URL: <URL>/v1
API Key:  sk-no-key-required
Model:    qwen3-14b-abliterated
```

**Aider:**
```bash
aider --model openai/qwen3-14b-abliterated \
      --openai-api-base <URL>/v1 \
      --openai-api-key sk-no-key-required
```

**OpenHands / Goose / Cline:**
همون Base URL + API key

### عیب‌یابی

- **سرعت پایین**: T4 رایگانه! ~15-25 tok/s برای generation. صبور باش.
- **قطع شد**: Colab رایگان بعد 1-2 ساعت idle میشه. دوباره همین نوت‌بوک رو اجرا کن.
- **OOM**: context رو کم کن (سلول 5: `CONTEXT_SIZE = 8192`)
- **URL باز نمیشه**: تونل گاهی کند میشه. سلول 7 رو دوباره اجرا کن.

### لاگ‌ها

```bash
!tail -20 /content/llama.log   # مدل
!tail -20 /content/argo.log    # Argo
!tail -20 /content/tunnel.log  # تونل
```